In [ ]:
pip install langchain-core langchain-google-genai langgraph sentence-transformers neo4j langchain-community numpy

In [ ]:
pip install langchain-groq

In [ ]:
!pip install ipython-autotime
%load_ext autotime

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase
from typing import TypedDict, Annotated
import numpy as np
import uuid
import os

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
from google.colab import userdata
api_go = userdata.get("Gemini")
api_gr = userdata.get("groqAPi")

In [ ]:
conv = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    temperature = 0.2,
    google_api_key = api_go
)

speed = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature = 0,
    groq_api_key = api_gr
)

### Relevancy tool


In [ ]:
@tool
def relevancy_ebd(player_message: str, recent_context: str) -> str:
    """
  When to call: before passing any player message to an NPC or
                taking any game action

  player_message: the exact text the player just sent

  recent_context: one sentence summarising the last 2 exchanges,
                  or "no prior context" if this is the first message

  output: yes if the player message is relevant to the recent context,
          no otherwise
    """
    embedding_player = embedder.encode(player_message)
    embedding_context = embedder.encode(recent_context)
    dot= np.dot(embedding_player, embedding_context) / (np.linalg.norm(embedding_player) * np.linalg.norm(embedding_context))
    if dot > 0.5:
        return f"yes \n dot value: {dot}"
    else:
        return f"no \n dot value: {dot}"




In [ ]:
@tool
def relevancy_llm(player_message: str, recent_context: str) -> str:
    """
  When to call: before passing any player message to an NPC or
                taking any game action

  player_message: the exact text the player just sent

  recent_context: one sentence summarising the last 2 exchanges,
                  or "no prior context" if this is the first message

  output: yes if the player message is relevant to the recent context,
          no otherwise
    """
    prompt = f"""You are a very experienced context manager
    you have to figure out based on previous discussion of the player with npc whether the current message is relevant or the player is just messing about

    here is the player message: {player_message}

    here is the recent context: {recent_context}

    STRICT OUTPUT CONDITION: only "yes" if the player message is relevant to the recent context,
          "no" otherwise, no justification or any other text
    """
    return speed.invoke(prompt)


In [ ]:
player_message = "We saw the missing page near the fireplace half burnt, it was near the table you were studying"
recent_context = "Where were you?\nI was in the study room\nDo you know about page 42?\nno, i saw the missing page when i entered thorne's room"

In [ ]:
%%time
ans = relevancy_llm.invoke({"player_message": player_message, "recent_context" : recent_context})
print(ans.content)

yes
CPU times: user 16.2 ms, sys: 2.05 ms, total: 18.2 ms
Wall time: 226 ms
time: 245 ms (started: 2026-03-21 06:37:44 +00:00)


In [ ]:
%%time
ans = ans = relevancy_ebd.invoke({"player_message": player_message, "recent_context" : recent_context})
print(ans)

yes 
 dot value: 0.6400977969169617
CPU times: user 61.6 ms, sys: 0 ns, total: 61.6 ms
Wall time: 62.2 ms
time: 63.3 ms (started: 2026-03-21 06:38:19 +00:00)
